<a href="https://colab.research.google.com/github/wilson121231/QRT-project-for-public/blob/main/%E5%9F%BA%E7%A1%80%E6%A8%A1%E5%9E%8B_%E7%AE%80%E5%8D%95%E7%BB%9F%E8%AE%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm import tqdm
# 显示所有列
pd.set_option("display.max_columns", None)


In [ ]:
# 加载数据
train_input = pd.read_csv("./data/X_train.csv")
train_output = pd.read_csv("./data/Y_train.csv")
test_input = pd.read_csv("./data/X_test.csv")

# 合并并创建标签
train_df = pd.merge(train_input, train_output, on="ID")
train_df["target"] = train_df["RET_TARGET"].apply(lambda x: 1 if x > 0 else -1)

# 处理缺失值
ret_cols = [col for col in train_input.columns if col.startswith("RET_")]
train_input[ret_cols] = train_input[ret_cols].fillna(0)
test_input[ret_cols] = test_input[ret_cols].fillna(0)

# 统计特征函数
def add_stat_features(df):
    df['ret_mean'] = df[ret_cols].mean(axis=1)
    df['ret_std'] = df[ret_cols].std(axis=1)
    df['ret_max'] = df[ret_cols].max(axis=1)
    df['ret_min'] = df[ret_cols].min(axis=1)
    df['ret_pos_ratio'] = (df[ret_cols] > 0).sum(axis=1) / len(ret_cols)
    return df

# 添加统计特征
train_df = add_stat_features(train_df)
test_input = add_stat_features(test_input)


C:\Users\6666s\AppData\Local\Temp\ipykernel_18488\1121256022.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ret_mean'] = df[ret_cols].mean(axis=1)
C:\Users\6666s\AppData\Local\Temp\ipykernel_18488\1121256022.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ret_std'] = df[ret_cols].std(axis=1)
C:\Users\6666s\AppData\Local\Temp\ipykernel_18488\1121256022.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

In [ ]:
# 对整个 DataFrame 所有的 NaN 填 0（适用于数值列）
train_df = train_df.fillna(0)
train_df.head()

,ID,ID_DAY,RET_216,RET_238,RET_45,RET_295,RET_230,RET_120,RET_188,RET_260,RET_15,RET_150,RET_229,RET_121,RET_156,RET_57,RET_203,RET_264,RET_58,RET_224,RET_30,RET_159,RET_236,RET_261,RET_88,RET_59,RET_242,RET_116,RET_84,RET_240,RET_97,RET_0,RET_256,RET_263,RET_40,RET_184,RET_105,RET_49,RET_99,RET_26,RET_262,RET_187,RET_168,RET_110,RET_123,RET_268,RET_48,RET_60,RET_226,RET_172,RET_197,RET_286,RET_182,RET_163,RET_63,RET_41,RET_201,RET_62,RET_35,RET_5,RET_223,RET_126,RET_87,RET_148,RET_266,RET_248,RET_55,RET_66,RET_74,RET_259,RET_37,RET_245,RET_222,RET_138,RET_31,RET_118,RET_181,RET_56,RET_250,RET_115,RET_18,RET_270,RET_213,RET_232,RET_275,RET_285,RET_83,RET_47,RET_265,RET_296,RET_276,RET_255,RET_108,RET_122,RET_194,RET_72,RET_293,RET_281,RET_193,RET_95,RET_162,RET_297,ID_TARGET,RET_TARGET,target,ret_mean,ret_std,ret_max,ret_min,ret_pos_ratio
0,0,3316,0.004024,0.009237,0.004967,0.0,0.01704,0.013885,0.041885,0.015207,-0.003143,0.018565,0.036312,0.002578,0.011782,0.009404,-0.024951,-0.005493,0.005786,0.005735,0.017187,0.040789,-0.006751,-0.013486,0.027113,0.007006,0.019158,0.012076,-0.006246,0.008563,-0.002689,-0.016501,0.029601,0.003583,0.031596,-0.022133,0.0,-0.010627,0.023026,0.011139,0.037835,0.004778,0.014719,0.057911,0.012849,0.079655,0.000439,0.045039,-0.005809,0.00514,-0.008609,0.0,-0.02878,0.037539,0.017845,0.022269,-0.004593,0.013131,-0.011702,0.070654,0.021528,0.043444,-0.008062,0.010045,0.014819,0.011586,-0.025717,0.001288,0.002289,0.033634,0.008878,0.008791,0.003914,-0.016817,0.018493,0.008937,-0.000624,0.032366,0.022977,0.021104,0.019672,0.004575,0.004258,0.00373,0.017005,-0.004232,0.032986,0.014069,0.026423,0.016533,0.011942,0.008573,0.027374,0.007596,0.01501,0.014733,-0.000476,0.006539,-0.010233,0.001251,-0.003102,-0.094847,139,-0.022351,-1,0.01039,0.021659,0.079655,-0.094847,0.73
1,1,3316,0.004024,0.009237,0.004967,0.0,0.01704,0.013885,0.041885,0.015207,-0.003143,0.018565,0.036312,0.002578,0.011782,0.009404,-0.024951,-0.005493,0.005786,0.005735,0.017187,0.040789,-0.006751,-0.013486,0.027113,0.007006,0.019158,0.012076,-0.006246,0.008563,-0.002689,-0.016501,0.029601,0.003583,0.031596,-0.022133,0.0,-0.010627,0.023026,0.011139,0.037835,0.004778,0.014719,0.057911,0.012849,0.079655,0.000439,0.045039,-0.005809,0.00514,-0.008609,0.0,-0.02878,0.037539,0.017845,0.022269,-0.004593,0.013131,-0.011702,0.070654,0.021528,0.043444,-0.008062,0.010045,0.014819,0.011586,-0.025717,0.001288,0.002289,0.033634,0.008878,0.008791,0.003914,-0.016817,0.018493,0.008937,-0.000624,0.032366,0.022977,0.021104,0.019672,0.004575,0.004258,0.00373,0.017005,-0.004232,0.032986,0.014069,0.026423,0.016533,0.011942,0.008573,0.027374,0.007596,0.01501,0.014733,-0.000476,0.006539,-0.010233,0.001251,-0.003102,-0.094847,129,-0.011892,-1,0.01039,0.021659,0.079655,-0.094847,0.73
2,2,3316,0.004024,0.009237,0.004967,0.0,0.01704,0.013885,0.041885,0.015207,-0.003143,0.018565,0.036312,0.002578,0.011782,0.009404,-0.024951,-0.005493,0.005786,0.005735,0.017187,0.040789,-0.006751,-0.013486,0.027113,0.007006,0.019158,0.012076,-0.006246,0.008563,-0.002689,-0.016501,0.029601,0.003583,0.031596,-0.022133,0.0,-0.010627,0.023026,0.011139,0.037835,0.004778,0.014719,0.057911,0.012849,0.079655,0.000439,0.045039,-0.005809,0.00514,-0.008609,0.0,-0.02878,0.037539,0.017845,0.022269,-0.004593,0.013131,-0.011702,0.070654,0.021528,0.043444,-0.008062,0.010045,0.014819,0.011586,-0.025717,0.001288,0.002289,0.033634,0.008878,0.008791,0.003914,-0.016817,0.018493,0.008937,-0.000624,0.032366,0.022977,0.021104,0.019672,0.004575,0.004258,0.00373,0.017005,-0.004232,0.032986,0.014069,0.026423,0.016533,0.011942,0.008573,0.027374,0.007596,0.01501,0.014733,-0.000476,0.006539,-0.010233,0.001251,-0.003102,-0.094847,136,-0.015285,-1,0.01039,0.021659,0.079655,-0.094847,0.73
3,3,3316,0.004024,0.009237,0.004967,0.0,0.01704,0.013885,0.041885,0.015207,-0.003143,0.018565,0.036312,0.002578,0.011782,0.009404,-0.024951,-0.005493,0.005786,0.005735,0.017187,0.040789,-0.006751,-0.013486,0.027113,0.007006,0.019158,0.0

In [ ]:
from sklearn.model_selection import train_test_split

train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,               # 20% 作为验证集
    random_state=42)            # 固定随机种子，保证每次划分一

In [ ]:
train_part.head()

,ID,ID_DAY,RET_216,RET_238,RET_45,RET_295,RET_230,RET_120,RET_188,RET_260,RET_15,RET_150,RET_229,RET_121,RET_156,RET_57,RET_203,RET_264,RET_58,RET_224,RET_30,RET_159,RET_236,RET_261,RET_88,RET_59,RET_242,RET_116,RET_84,RET_240,RET_97,RET_0,RET_256,RET_263,RET_40,RET_184,RET_105,RET_49,RET_99,RET_26,RET_262,RET_187,RET_168,RET_110,RET_123,RET_268,RET_48,RET_60,RET_226,RET_172,RET_197,RET_286,RET_182,RET_163,RET_63,RET_41,RET_201,RET_62,RET_35,RET_5,RET_223,RET_126,RET_87,RET_148,RET_266,RET_248,RET_55,RET_66,RET_74,RET_259,RET_37,RET_245,RET_222,RET_138,RET_31,RET_118,RET_181,RET_56,RET_250,RET_115,RET_18,RET_270,RET_213,RET_232,RET_275,RET_285,RET_83,RET_47,RET_265,RET_296,RET_276,RET_255,RET_108,RET_122,RET_194,RET_72,RET_293,RET_281,RET_193,RET_95,RET_162,RET_297,ID_TARGET,RET_TARGET,target,ret_mean,ret_std,ret_max,ret_min,ret_pos_ratio
30821,30821,1186,0.002462,-0.013818,-0.011011,-0.037329,-0.013917,-0.007549,0.000000,-0.002661,-0.016703,-0.021520,-0.005219,-0.007551,0.011754,-0.020300,0.001613,-0.014538,-0.017677,-0.012774,0.011491,0.096460,0.000000,-0.012876,0.000000,0.000000,-0.016081,-0.000114,-0.015671,0.048527,0.000000,-0.006123,-0.016039,-0.003254,-0.009612,-0.033805,-0.006648,-0.027920,-0.013882,-0.017310,-0.024363,0.020070,0.006676,-0.020268,-0.013046,-0.005896,0.003552,-0.017522,-0.005291,-0.003852,0.000000,-0.016863,-0.011291,-0.000149,-0.003534,0.022298,-0.014616,-0.004094,-0.010874,0.011987,-0.007052,-0.001507,0.000286,-0.005190,-0.011834,-0.004999,0.013443,-0.005826,-0.017504,0.009801,-0.001407,0.007728,0.019585,-0.014326,-0.015833,-0.006250,-0.016645,0.000000,-0.017708,-0.011799,-0.013120,-0.017845,-0.008383,-0.007218,-0.002857,0.036048,-0.016083,-0.021769,-0.002851,-0.010152,0.020272,0.010864,-0.028200,-0.005390,-0.007536,0.015774,-0.006059,-0.014885,-0.020763,-0.002882,0.000000,0.026744,132,-0.002561,-1,-0.005000,0.017807,0.096460,-0.037329,0.21
152261,152261,2423,0.007108,-0.017057,0.017228,-0.029882,-0.023199,0.006929,-0.020037,-0.017012,-0.026529,-0.051192,-0.016715,0.048704,-0.018809,-0.004391,-0.009689,0.002007,-0.014764,-0.046207,-0.012662,-0.081439,-0.024616,-0.023816,-0.041813,-0.017523,-0.026848,-0.014273,0.010563,-0.014297,-0.052618,-0.022769,-0.006459,-0.019468,-0.013055,-0.003078,0.002273,-0.050153,-0.033640,-0.031286,0.009794,-0.008969,-0.034468,0.040883,-0.004418,-0.056833,-0.050509,0.013848,-0.017878,-0.030033,-0.030405,-0.000396,-0.011855,-0.013251,-0.045163,-0.001319,-0.035125,-0.016557,-0.010286,-0.017777,0.001362,0.001022,-0.030049,-0.008590,-0.029058,-0.013555,-0.021965,-0.004827,-0.026713,0.027160,-0.013390,-0.055663,-0.016839,-0.017879,-0.010829,-0.029181,0.000332,-0.015871,-0.010797,-0.016454,0.006286,-0.038884,-0.009264,0.000000,-0.028216,-0.044813,0.024737,-0.009473,0.031246,-0.009590,-0.018615,-0.007137,-0.019758,-0.031895,-0.009635,0.005386,-0.005928,-0.033931,-0.009321,0.006693,-0.010079,-0.023595,109,-0.013326,-1,-0.015543,0.020825,0.048704,-0.081439,0.19
195712,195712,2461,0.015411,-0.094362,0.008378,0.010537,-0.014777,0.007037,0.011294,0.012976,0.006098,0.015305,0.018912,-0.012678,0.008358,0.007430,-0.008044,0.015498,0.010707,0.007871,0.015251,0.000000,0.004845,0.013034,0.019084,0.013393,0.000000,0.013382,0.008376,0.006636,0.008550,0.002096,0.011895,-0.025437,0.014215,0.031378,-0.003763,0.046529,0.000614,0.010794,0.021684,0.008915,0.013182,0.004100,0.003208,-0.000356,0.010471,0.006643,0.016345,0.027546,0.003062,-0.013494,0.020086,0.000012,-0.002322,0.012301,-0.003118,0.017834,0.000070,0.011839,0.004366,-0.025708,0.006375,0.001671,0.011697,-0.008817,0.019038,0.018351,0.037483,0.047469,0.014692,0.018893,0.007710,0.007437,0.009374,0.013732,-0.012653,-0.005477,0.012975,0.021859,-0.002419,0.011662,-0.002605,0.027666,0.010718,0.009057,0.009179,-0.035192,-0.036725,0.012182,0.004204,0.049829,0.021141,0.003452,-0.000572,0.010379,0.005071,0.012462,0.001474,0.016798,0.006074,0.042951,8,-0.000763,-1,0.007674,0.017794,0.049829,-0.094362,0.79
176485,176485,3869,0.014450,0.024880,0.005237,0

In [ ]:
# 选择特征列（排除非输入字段）
X_cols = [
    col for col in train_df.columns
    if col not in ['ID_TARGET', 'target', 'RET_TARGET']  # 排除标签和泄露字段
    and not col.startswith('CLASS_LEVEL')                # 若不使用行业分类
    and col not in ['degree', 'clustering', 'pagerank']  # 若不使用图结构特征
]

In [ ]:
import lightgbm as lgb
from tqdm import tqdm

models = {}
val_preds = []


# 分目标资产进行训练和预测
for target in tqdm(train_df["ID_TARGET"].unique()):
    sub_train = train_part[train_part["ID_TARGET"] == target]
    sub_val = val_part[val_part["ID_TARGET"] == target]

    if sub_train.empty or sub_val.empty:
        continue

    X_train = sub_train[X_cols]
    y_train = (sub_train["target"] == 1).astype(int)

    # 类别不够时无法训练
    if y_train.nunique() < 2:
        continue

    X_val = sub_val[X_cols]
    y_val_true = sub_val["target"]

    # LightGBM 模型
    model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    models[target] = model

    # 验证集预测
    y_val_pred = model.predict(X_val)
    y_val_pred_sign = [1 if p == 1 else -1 for p in y_val_pred]

    # 收集结果
    val_preds.extend(zip(y_val_true, y_val_pred_sign))

# 准确率函数
def calculate_accuracy(val_preds):
    correct = sum(yt == yp for yt, yp in val_preds)
    total = len(val_preds)
    return correct / total if total > 0 else 0

# 输出
acc = calculate_accuracy(val_preds)
print(f"✅ LightGBM 验证集准确率（分目标资产训练）: {acc:.4f}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tqdm import tqdm

models_rf = {}
val_preds_rf = []


# 对每个目标股票单独训练模型
for target in tqdm(train_df["ID_TARGET"].unique()):
    sub_df = train_df[train_df["ID_TARGET"] == target]

    if sub_df["target"].nunique() < 2 or len(sub_df) < 10:
        continue  # 样本太少或类别单一，跳过

    # 用该股票自身的数据划分训练/验证（按日期随机）
    train_sub, val_sub = train_test_split(sub_df, test_size=0.2, random_state=42, stratify=sub_df["target"])

    X_train = train_sub[X_cols]
    y_train = (train_sub["target"] == 1).astype(int)

    X_val = val_sub[X_cols]
    y_val_true = val_sub["target"]

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    models_rf[target] = model

    y_val_pred = model.predict(X_val)
    y_val_pred_sign = [1 if p == 1 else -1 for p in y_val_pred]
    val_preds_rf.extend(zip(y_val_true, y_val_pred_sign))

# 准确率
def calculate_accuracy(val_preds):
    correct = sum(yt == yp for yt, yp in val_preds)
    total = len(val_preds)
    return correct / total if total > 0 else 0

acc_rf = calculate_accuracy(val_preds_rf)
print(f"✅ Random Forest 验证集准确率（每股票内部划分验证集）: {acc_rf:.4f}")


100%|██████████| 100/100 [01:45<00:00,  1.05s/it]

✅ Random Forest 验证集准确率（每股票内部划分验证集）: 0.6594


In [ ]:
X_test=pd.read_csv("./data/X_test.csv")
X_test = add_stat_features(X_test)
# ⬇️ 用 RF 模型对 X_test 生成 submission 文件
test_preds = {}

for idx, row in tqdm(X_test.iterrows(), total=X_test.shape[0]):
    target = row["ID_TARGET"]
    if target not in models_rf:
        continue
    model = models_rf[target]

    # 构造测试特征
    row_features = pd.DataFrame([row[X_cols].values], columns=X_cols)

    # 预测并转换为方向（-1 or 1）
    pred = model.predict(row_features)[0]
    test_preds[row["ID"]] = 1 if pred == 1 else -1

# 构建 DataFrame
submission = pd.DataFrame.from_dict(test_preds, orient="index").reset_index()
submission.columns = ["ID", "RET_TARGET"]

# 保证 ID 顺序一致（严格对齐 X_test）
submission = submission.set_index("ID").reindex(X_test["ID"]).reset_index()

# 检查缺失
if submission["RET_TARGET"].isna().any():
    print("⚠️ 存在未预测的样本，用 0 填充")
    submission["RET_TARGET"] = submission["RET_TARGET"].fillna(0)

# 保存为 CSV
submission.to_csv("submission_rf.csv", index=False)
print("✅ 已生成 submission_rf.csv，可提交")

